# YOLOv8m — Esophagitis Detection
**Dataset:** Roboflow `abosare3/esophigitis` v2  
**Model:** YOLOv8 medium (pretrained on COCO)  

> ⚠️ Before running: go to **Session options → Accelerator → GPU T4 x2** and turn **Internet ON**

## Cell 1 — Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q

## Cell 2 — Verify GPU

In [ ]:
import torch
print("CUDA available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — check accelerator settings!")
print("PyTorch version:", torch.__version__)

## Cell 3 — Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="srOlHtgX9lDpElVHhUQ3")
project = rf.workspace("abosare3").project("esophigitis")
version = project.version(2)
dataset = version.download("yolov8")

print("\nDataset downloaded to:", dataset.location)

## Cell 4 — Fix data.yaml Paths

In [ ]:
import yaml
from pathlib import Path

dataset_root = Path(dataset.location).resolve()
data_yaml    = dataset_root / "data.yaml"

with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

# Resolve relative paths to absolute so YOLO finds them regardless of cwd
for key in ("train", "val", "test"):
    if key in cfg and cfg[key] is not None:
        p = Path(cfg[key])
        if not p.is_absolute():
            resolved = (dataset_root / p).resolve()
            if resolved.exists():
                cfg[key] = str(resolved)
            else:
                direct = (dataset_root / key / "images").resolve()
                if direct.exists():
                    cfg[key] = str(direct)

patched_yaml = dataset_root / "data_abs.yaml"
with open(patched_yaml, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Dataset root :", dataset_root)
print("Patched yaml :", patched_yaml)
print("Classes      :", cfg.get("names"), "| nc =", cfg.get("nc"))
print("Train path   :", cfg.get("train"))
print("Val path     :", cfg.get("val"))

## Cell 5 — Configure Training

In [ ]:
# ── Hyper-parameters ──────────────────────────────────────
WEIGHTS      = "yolov8m.pt"   # pretrained COCO weights
EPOCHS       = 100
IMG_SIZE     = 640
BATCH_SIZE   = 16             # T4 has 16GB VRAM — safe at 16
PATIENCE     = 20             # early stopping
WORKERS      = 2              # Kaggle recommended
DEVICE       = 0              # GPU 0
PROJECT_DIR  = "/kaggle/working/runs/esophagitis"
RUN_NAME     = "yolov8m_v1"

print("Config ready. Proceed to training cell.")

## Cell 6 — Train

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS)

results = model.train(
    data        = str(patched_yaml),
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    patience    = PATIENCE,
    workers     = WORKERS,
    device      = DEVICE,
    project     = PROJECT_DIR,
    name        = RUN_NAME,

    # Augmentation
    hsv_h       = 0.015,
    hsv_s       = 0.4,
    hsv_v       = 0.4,
    flipud      = 0.5,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,

    # Saving
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,
)

print("\nTraining complete!")

## Cell 7 — Validate (Best Checkpoint)

In [ ]:
from pathlib import Path

best_weights = Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
best_model   = YOLO(str(best_weights))

val_results = best_model.val(
    data   = str(patched_yaml),
    imgsz  = IMG_SIZE,
    batch  = BATCH_SIZE,
    device = DEVICE,
    split  = "val",
)

print("\n" + "="*50)
print("VALIDATION RESULTS")
print("="*50)
print(f"mAP50      : {val_results.box.map50:.4f}")
print(f"mAP50-95   : {val_results.box.map:.4f}")
print(f"Precision  : {val_results.box.mp:.4f}")
print(f"Recall     : {val_results.box.mr:.4f}")

## Cell 8 — Show Training Plots

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path(PROJECT_DIR) / RUN_NAME

for plot in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = run_dir / plot
    if p.exists():
        print(f"\n── {plot} ──")
        display(Image(str(p), width=900))

## Cell 9 — Save Best Model to Kaggle Output

In [ ]:
import shutil
from pathlib import Path

run_dir      = Path(PROJECT_DIR) / RUN_NAME
best_src     = run_dir / "weights" / "best.pt"
best_dst     = Path("/kaggle/working/best.pt")

shutil.copy(best_src, best_dst)

print("Best model saved to:", best_dst)
print("Download it from the Kaggle output panel on the right →")